# 📊 Análise de Componentes Principais (PCA) para Elaboração de Rankings Socioeconômicos

## 📌 Contexto do Projeto
O objetivo deste projeto é elaborar um ranking dos distritos do município de São Paulo, ordenando-os com base em indicadores socioeconômicos coletados pela prefeitura. A base de dados contém múltiplas variáveis numéricas e, para evitar uma análise isolada ou arbitrária baseada em apenas um único indicador, é utilizada a técnica de **PCA (Principal Component Analysis)** para consolidar todas as informações simultaneamente em um índice estruturado.

---

## 🧠 Fundamentos Teóricos do PCA

### 1. Definição da Técnica
O PCA é uma técnica de **aprendizado não-supervisionado**. Isso significa que os dados não possuem uma variável alvo (*target*) predefinida. Em vez de realizar previsões de valores futuros, o método executa uma **transformação matemática** nos dados originais, convertendo as variáveis em **fatores independentes (componentes principais)**.

* **Ordenação de Variância:** O primeiro fator gerado explica a maior parcela da variabilidade dos dados. O segundo fator retém a segunda maior parcela, e assim sucessivamente.
* **Limite de Componentes:** A quantidade máxima de fatores gerados é exatamente igual ao número de variáveis numéricas originais presentes no conjunto de dados.

### 2. Principais Aplicações do PCA
* **Redução de Dimensionalidade:** Substitui um grande volume de variáveis originais por apenas alguns fatores principais, reduzindo o tamanho do dataset sem perda significativa de informação.
* **Remoção de Multicolinearidade:** Elimina problemas onde as variáveis originais são excessivamente correlacionadas entre si. Os novos fatores gerados são totalmente independentes (ortogonais) e não possuem correlação mútua.
* **Elaboração de Rankings:** Permite consolidar múltiplos indicadores complexos em um índice único de pontuação para ordenação de observações.

### ⚠️ Restrições Técnicas
O algoritmo do PCA fundamenta-se estritamente na construção de uma **Matriz de Correlação**. Por consequência, a técnica **aceita apenas variáveis numéricas**. Variáveis categóricas ou textuais (como identificadores e nomes de regiões) devem ser isoladas do cálculo estatístico principal.


In [7]:
import pandas as pd


In [ ]:
distritos = pd.read_csv("Dados/distritos_sp.csv")


In [9]:
distritos


,cod_ibge,distritos,renda,quota,escolaridade,idade,mortalidade,txcresc,causasext,favel,denspop
0,1,Água Rasa,1961,34.619999,7.6,32,13.86,-1.840000,52.980000,0.00,125.610001
1,12,Alto de Pinheiros,4180,75.959999,8.4,33,8.68,-2.520000,38.570000,0.69,57.560001
2,23,Anhanguera,1093,4.500000,5.8,23,15.36,18.120001,22.680000,0.00,8.570000
3,34,Aricanduva,1311,21.020000,6.8,27,18.43,-1.070000,76.220001,5.38,138.539993
4,45,Artur Alvim,1248,15.910000,7.0,27,19.73,-1.400000,67.250000,4.11,167.399994
...,...,...,...,...,...,...,...,...,...,...,...
91,92,Vila Medeiros,1405,19.760000,6.8,27,15.43,-1.410000,77.980003,2.49,188.929993
92,93,Vila Prudente,1755,32.080002,7.2,30,14.36,-2.550000,66.510002,7.43,101.440002
93,94,Vila Sônia,2970,41.410000,7.4,27,16.76,-0.900000,74.680000,14.93,80.120003
94,95,São Domingos,2047,23.510000,6.8,26,14.30,0.710000,62.349998,8.55,72.919998


In [10]:
distritos.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   cod_ibge      96 non-null     int64  
 1   distritos     96 non-null     str    
 2   renda         96 non-null     int64  
 3   quota         96 non-null     float64
 4   escolaridade  96 non-null     float64
 5   idade         96 non-null     int64  
 6   mortalidade   96 non-null     float64
 7   txcresc       96 non-null     float64
 8   causasext     96 non-null     float64
 9   favel         96 non-null     float64
 10  denspop       96 non-null     float64
dtypes: float64(7), int64(3), str(1)
memory usage: 8.4 KB


In [12]:
variaveis_numericas = distritos.drop(['cod_ibge', 'distritos'], axis=1)
variaveis_numericas.corr()

,renda,quota,escolaridade,idade,mortalidade,txcresc,causasext,favel,denspop
renda,1.000000,0.920099,0.777332,0.732307,-0.519585,-0.424711,-0.462516,-0.146957,-0.019711
quota,0.920099,1.000000,0.850455,0.832737,-0.520282,-0.554767,-0.491020,-0.243010,0.057374
escolaridade,0.777332,0.850455,1.000000,0.955825,-0.582601,-0.692968,-0.606621,-0.432548,0.157673
idade,0.732307,0.832737,0.955825,1.000000,-0.553758,-0.703237,-0.615073,-0.499838,0.141469
mortalidade,-0.519585,-0.520282,-0.582601,-0.553758,1.000000,0.346049,0.422790,0.130877,-0.093018
txcresc,-0.424711,-0.554767,-0.692968,-0.703237,0.346049,1.000000,0.234472,0.281853,-0.279084
causasext,-0.462516,-0.491020,-0.606621,-0.615073,0.422790,0.234472,1.000000,0.404447,-0.045281
favel,-0.146957,-0.243010,-0.432548,-0.499838,0.130877,0.281853,0.404447,1.000000,-0.106481
denspop,-0.019711,0.057374,0.157673,0.141469,-0.093018,-0.279084,-0.045281,-0.106481,1.000000


# 📉 Calculando a Estatística KMO (Kaiser-Meyer-Olkin)

## 📌 O que é o Critério KMO?
O teste de **Kaiser-Meyer-Olkin (KMO)** é um índice estatístico que avalia se o conjunto de dados é adequado para a aplicação de uma análise fatorial ou PCA. Ele mede a consistência geral dos dados, indicando a proporção da variância das variáveis que pode ser considerada comum (atribuída a fatores compartilhados).

---

## 📊 Interpretação dos Resultados
A estatística KMO varia estritamente em um intervalo de **0 a 1**:

* **Próximos de 1:** Indicam que as variáveis compartilham um percentual de variância bastante elevado (correlações de Pearson altas). Significa que os dados estão **adequadamente ajustados** para o modelo.
* **Próximos de 0:** Decorrem de correlações de Pearson baixas entre as variáveis. Isso serve como um sinal de alerta, indicando que a análise fatorial ou PCA será **inadequada** para esse conjunto de dados.


In [14]:
from factor_analyzer.factor_analyzer import calculate_kmo


In [15]:
kmo_variaveis, kmo = calculate_kmo(variaveis_numericas)

In [16]:
kmo_variaveis

array([0.77821831, 0.81941916, 0.8560973 , 0.81750459, 0.94677797,
       0.84146713, 0.89083164, 0.78871213, 0.63275248])

In [17]:
kmo

np.float64(0.833091424182929)

# 📉 Teste de Esfericidade de Bartlett

## 📌 Objetivo do Teste
O **Teste de Esfericidade de Bartlett** avalia a adequabilidade dos dados para a análise fatorial ou PCA. Ele faz isso comparando a **matriz de correlações ($\rho$)** das variáveis originais com uma **matriz identidade ($I$)** de mesma dimensão.

* **Matriz Identidade ($I$):** É uma matriz que possui o número 1 na diagonal principal e zero em todas as outras posições. Caso nossos dados fossem iguais a ela, significaria que a correlação entre as variáveis é nula.

---

## 🔬 Hipóteses Estatísticas
O teste avalia duas hipóteses principais:

* **Hipótese Nula ($H_0$):** $\rho = I$
  A matriz de correlações é igual à matriz identidade. Indica que as variáveis **não possuem correlação significativa** entre si. Se não há correlação, a extração de fatores do PCA **não será adequada**.
* **Hipótese Alternativa ($H_1$):** $\rho \neq I$
  A matriz de correlações é significativamente diferente da matriz identidade. Indica que as variáveis **possuem correlações significativas** entre si, validando a aplicação do PCA.

---

## 🎯 Regra de Decisão (O que buscamos?)
Para podermos seguir com o PCA, o nosso objetivo é **rejeitar a Hipótese Nula ($H_0$)**. 

Na prática do código, isso é avaliado através do **p-valor (p-value)**. Buscamos um **p-valor menor que 0.05** (considerando um nível de significância clássico de 5%), o que comprova que as correlações fora da diagonal principal são estatisticamente diferentes de zero.

---

## ⚠️ Observação Metodológica Crucial
Deve-se **sempre preferir o Teste de Esfericidade de Bartlett à estatística KMO** para fins de decisão definitiva sobre a adequação global da análise fatorial ou PCA. 

* **Bartlett:** É um teste de hipóteses formal, fundamentado em uma distribuição de probabilidades determinada e níveis rígidos de significância estatística.
* **KMO:** Funciona apenas como um coeficiente (estatística descritiva), calculado sem uma distribuição de probabilidades associada, o que impossibilita a avaliação formal de um nível de significância correspondente para a tomada de decisão.


In [19]:
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity


In [20]:
qui_quadrado, p_valor = calculate_bartlett_sphericity(variaveis_numericas)

In [21]:
print(qui_quadrado)
print(p_valor)

748.1593126421544
5.607017481839493e-134


In [23]:
p_valor < 0.05


np.True_

## 📐 Aprofundamento Matemático: A Equação Característica ($Av = \lambda v$)

O PCA realiza uma transformação linear dos dados para um Crypto-sistema de coordenadas. Nesse processo, os autovetores ($v$) representam direções espaciais especiais, e os autovalores ($\lambda$) representam fatores de escala.

A relação fundamental entre eles é expressa pela equação:
$$Av = \lambda v$$

Onde:
* **$A$**: É a matriz de transformação (no caso do PCA, a **Matriz de Correlação** das variáveis).
* **$v$**: É o **Autovetor**.
* **$\lambda$**: É o **Autovalor** correspondente.

Isso significa que aplicar a transformação linear da matriz de correlação sobre o autovetor gera o mesmo resultado matemático que simplesmente multiplicar esse autovetor pelo número escalar $\lambda$. Por resumirem as propriedades essenciais da matriz de correlação, eles são a base para transformar as variáveis originais nos fatores finais.

---

### 🔬 Como encontrar os Autovalores e Autovetores?

Podemos reescrever a equação fundamental isolando os termos:
$$Av - \lambda v = 0$$

Para subtrair um número escalar ($\lambda$) de uma matriz ($A$), multiplicamos o escalar pela **Matriz Identidade ($I$)**, resultando na equação característica do sistema:
$$(A - \lambda I)v = 0$$

1. **Encontrando os Autovalores ($\lambda$):** 
   Para que o sistema possua uma solução real (onde $v \neq 0$), a matriz $(A - \lambda I)$ precisa ser **singular** (não-invertível). Por propriedade matemática, uma matriz singular possui determinante igual a zero:
   $$det(A - \lambda I) = 0$$
   Ao resolver esse determinante, obtemos um polinômio cujas raízes são os autovalores ($\lambda$) do projeto.

2. **Encontrando os Autovetores ($v$):**
   Com os autovalores calculados, basta substituir cada valor de $\lambda$ de volta na expressão $(A - \lambda I)v = 0$ para resolver o sistema de equações e encontrar o vetor $v$ associado a cada um.


In [24]:
variaveis_numericas

,renda,quota,escolaridade,idade,mortalidade,txcresc,causasext,favel,denspop
0,1961,34.619999,7.6,32,13.86,-1.840000,52.980000,0.00,125.610001
1,4180,75.959999,8.4,33,8.68,-2.520000,38.570000,0.69,57.560001
2,1093,4.500000,5.8,23,15.36,18.120001,22.680000,0.00,8.570000
3,1311,21.020000,6.8,27,18.43,-1.070000,76.220001,5.38,138.539993
4,1248,15.910000,7.0,27,19.73,-1.400000,67.250000,4.11,167.399994
...,...,...,...,...,...,...,...,...,...
91,1405,19.760000,6.8,27,15.43,-1.410000,77.980003,2.49,188.929993
92,1755,32.080002,7.2,30,14.36,-2.550000,66.510002,7.43,101.440002
93,2970,41.410000,7.4,27,16.76,-0.900000,74.680000,14.93,80.120003
94,2047,23.510000,6.8,26,14.30,0.710000,62.349998,8.55,72.919998


In [25]:
from sklearn.preprocessing import StandardScaler

In [26]:
colunas_numericas = variaveis_numericas.columns
colunas_numericas

Index(['renda', 'quota', 'escolaridade', 'idade', 'mortalidade', 'txcresc',
       'causasext', 'favel', 'denspop'],
      dtype='str')

In [28]:
padronizar = StandardScaler()
variaveis_numericas = padronizar.fit_transform(variaveis_numericas)
variaveis_numericas 

array([[ 1.08502328e-01,  1.80715140e-01,  5.23096244e-01,
         1.05541274e+00, -4.29281808e-01, -5.38805723e-01,
        -7.80681293e-01, -7.56406323e-01,  5.21804668e-01],
       [ 2.40630528e+00,  1.98438577e+00,  1.32019516e+00,
         1.28464265e+00, -1.47188596e+00, -7.33574170e-01,
        -1.40197826e+00, -6.68425227e-01, -8.57183702e-01],
       [-7.90322801e-01, -1.13342509e+00, -1.27037668e+00,
        -1.00765651e+00, -1.27369371e-01,  5.17822174e+00,
        -2.08708641e+00, -7.56406323e-01, -1.84993379e+00],
       [-5.64581006e-01, -4.12654904e-01, -2.74002674e-01,
        -9.07368417e-02,  4.90544879e-01, -3.18259087e-01,
         2.21327218e-01, -7.04087705e-02,  7.83822446e-01],
       [-6.29818314e-01, -6.35605024e-01, -7.47280632e-02,
        -9.07368417e-02,  7.52202169e-01, -4.12779054e-01,
        -1.65420498e-01, -2.32344990e-01,  1.36865133e+00],
       [ 5.20636431e-01,  1.67189899e-01,  9.21645940e-01,
         8.26182822e-01, -1.48396254e+00, -6.247330

In [29]:
dados_padronizados = pd.DataFrame(variaveis_numericas, columns = colunas_numericas)
dados_padronizados

,renda,quota,escolaridade,idade,mortalidade,txcresc,causasext,favel,denspop
0,0.108502,0.180715,0.523096,1.055413,-0.429282,-0.538806,-0.780681,-0.756406,0.521805
1,2.406305,1.984386,1.320195,1.284643,-1.471886,-0.733574,-1.401978,-0.668425,-0.857184
2,-0.790323,-1.133425,-1.270377,-1.007657,-0.127369,5.178222,-2.087086,-0.756406,-1.849934
3,-0.564581,-0.412655,-0.274003,-0.090737,0.490545,-0.318259,0.221327,-0.070409,0.783822
4,-0.629818,-0.635605,-0.074728,-0.090737,0.752202,-0.412779,-0.165420,-0.232345,1.368651
...,...,...,...,...,...,...,...,...,...
91,-0.467243,-0.467629,-0.274003,-0.090737,-0.113280,-0.415643,0.297211,-0.438909,1.804943
92,-0.104813,0.069895,0.124547,0.596953,-0.328644,-0.742167,-0.197326,0.190984,0.032016
93,1.153335,0.476964,0.323822,-0.090737,0.154416,-0.269567,0.154929,1.147301,-0.400020
94,0.197556,-0.304016,-0.274003,-0.319967,-0.340721,0.191576,-0.376687,0.333794,-0.545923


In [30]:
from sklearn.decomposition import PCA

In [31]:
n_fatores = dados_padronizados.shape[1]
n_fatores

9

In [33]:
pca = PCA(n_components = n_fatores)
pca.fit(dados_padronizados)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",9
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 